In [11]:
pip install transformers

In [12]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import cv2
import random
from transformers import ViTModel, ViTFeatureExtractor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [13]:
feature_extractor = ViTFeatureExtractor.from_pretrained("google/vit-base-patch16-224-in21k")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=feature_extractor.image_mean, std=feature_extractor.image_std)
])

C:\Users\rimjh\anaconda3\Lib\site-packages\transformers\models\vit\feature_extraction_vit.py:28: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(


In [15]:
class VideoDatasetViT(Dataset):
    def __init__(self, root_dir, transform=None, num_frames=5):
        self.real_dir = os.path.join(root_dir, "videos_real")
        self.fake_dir = os.path.join(root_dir, "videos_fake")
        self.transform = transform
        self.num_frames = num_frames

        self.samples = [(os.path.join(self.real_dir, f), 0) for f in os.listdir(self.real_dir)]
        self.samples += [(os.path.join(self.fake_dir, f), 1) for f in os.listdir(self.fake_dir)]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        frames = self._extract_frames(video_path)
        return frames, torch.tensor(label, dtype=torch.float32)

    def _extract_frames(self, video_path):
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        indices = sorted(random.sample(range(total_frames), min(self.num_frames, total_frames)))

        frames = []
        for i in range(total_frames):
            ret, frame = cap.read()
            if not ret:
                break
            if i in indices:
                img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                img = Image.fromarray(img)
                if self.transform:
                    img = self.transform(img)
                frames.append(img)
        cap.release()
        if len(frames) < self.num_frames:
            frames += [frames[-1]] * (self.num_frames - len(frames))
        return torch.stack(frames)

In [16]:
dataset_path = r"C:\Users\rimjh\Downloads\dataset"
dataset = VideoDatasetViT(dataset_path, transform=transform)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [21]:
class ViTClassifier(nn.Module):
    def __init__(self):
        super(ViTClassifier, self).__init__()
        self.vit = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
        self.classifier = nn.Linear(self.vit.config.hidden_size, 1)

    def forward(self, x):
        # x shape: (batch_size, num_frames, C, H, W)
        batch_size, num_frames, C, H, W = x.shape
        x = x.view(-1, C, H, W)  # reshape to (batch*num_frames, C, H, W)
        outputs = self.vit(pixel_values=x).last_hidden_state[:, 0, :]  # CLS token
        outputs = outputs.view(batch_size, num_frames, -1)
        avg_output = outputs.mean(dim=1)
        return self.classifier(avg_output)

In [22]:
model = ViTClassifier().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

In [23]:
def train(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for frames, labels in dataloader:
        frames, labels = frames.to(device), labels.to(device).unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [24]:
def train(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for frames, labels in dataloader:
        frames, labels = frames.to(device), labels.to(device).unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(frames)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [25]:
def evaluate(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for frames, labels in dataloader:
            frames, labels = frames.to(device), labels.to(device).unsqueeze(1)
            outputs = torch.sigmoid(model(frames))
            preds = (outputs > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [26]:
for epoch in range(10):
    loss = train(model, dataloader, optimizer, criterion, device)
    acc = evaluate(model, dataloader, device)
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}, Accuracy: {acc:.4f}")

Epoch 1, Loss: 0.7188, Accuracy: 0.5566
Epoch 2, Loss: 0.6945, Accuracy: 0.7642
Epoch 3, Loss: 0.6634, Accuracy: 0.7925
Epoch 4, Loss: 0.6270, Accuracy: 0.7642
Epoch 5, Loss: 0.6052, Accuracy: 0.6887
Epoch 6, Loss: 0.5109, Accuracy: 0.9057
Epoch 7, Loss: 0.5300, Accuracy: 0.8396
Epoch 8, Loss: 0.4084, Accuracy: 0.8868
Epoch 9, Loss: 0.3224, Accuracy: 0.9434
Epoch 10, Loss: 0.3040, Accuracy: 0.8679


In [27]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import torch
import numpy as np

def evaluate_metrics(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for videos, labels in dataloader:
            videos = videos.to(device)
            labels = labels.to(device)

            outputs = model(videos)
            preds = torch.sigmoid(outputs).cpu().numpy()
            preds = np.round(preds)  # Convert to 0 or 1

            all_preds.extend(preds.flatten())
            all_labels.extend(labels.cpu().numpy().flatten())

    # Convert to numpy arrays
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    # Calculate metrics
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    cm = confusion_matrix(all_labels, all_preds)

    print("✅ Evaluation Metrics:")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print("Confusion Matrix:")
    print(cm)

In [28]:
evaluate_metrics(model, dataloader, device)

✅ Evaluation Metrics:
Accuracy:  0.8868
Precision: 1.0000
Recall:    0.7736
F1 Score:  0.8723
Confusion Matrix:
[[53  0]
 [12 41]]


In [37]:
import cv2
import torch
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
def predict_video(model, device):
    import torchvision.transforms as transforms
    import cv2
    import torch
    import numpy as np

    video_path = input("📁 Enter the full path of the video to check if it's real or fake:\n")
    cap = cv2.VideoCapture(video_path)
    
    frames = []
    frame_count = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame_count >= 16:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
        frame_count += 1
    cap.release()

    if len(frames) == 0:
        print("🚫 No frames found in the video.")
        return

    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])

    # Apply transform to all frames and stack them
    input_frames = torch.stack([transform(frame) for frame in frames])  # (num_frames, C, H, W)
    input_frames = input_frames.unsqueeze(0).to(device)  # (1, num_frames, C, H, W)

    model.eval()
    with torch.no_grad():
        output = model(input_frames)
        prob = torch.sigmoid(output).item()

    print(f"\n🔍 Prediction Confidence (Real=0 → Fake=1): {prob:.4f}")
    if prob >= 0.5:
        print("🧠 This video is likely a 🔥 FAKE 🔥.")
    else:
        print("✅ This video appears to be REAL.")

In [39]:
print("\n✅ Now let's test a new video...")
predict_video(model, device)


✅ Now let's test a new video...


📁 Enter the full path of the video to check if it's real or fake:
 C:\Users\rimjh\Downloads\dataset\videos_fake\vs6.mp4



🔍 Prediction Confidence (Real=0 → Fake=1): 0.7709
🧠 This video is likely a 🔥 FAKE 🔥.
